In [ ]:
import os
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reproduce.py").is_file())
os.chdir(ROOT)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROJECT_PATH = Path("data/genomics")

In [ ]:
meta = pd.read_csv(PROJECT_PATH / "reference/metadata_complete.csv")
meta['population'] = meta['Strain'] + '_' + meta['Culture'].astype(str).str.zfill(2)
meta

In [ ]:
meta.query('population=="PLAC_09"')

In [ ]:
dfs = []

for i, row in meta.iterrows():
    df = pd.read_csv(PROJECT_PATH / f"out/{row['FolderDate']}/{row['source_file']}/output/output.gd.tsv", sep='\t')
    df["Strain"] = row['Strain']
    df["Culture"] = row['Culture']
    df["Day"] = int(row['Day'])
    dfs.append(df)

df = pd.concat(dfs)
df



In [ ]:
select_lineage = "PLAC"
pops = meta.query(f'Strain=="{select_lineage}"')['population'].unique()
timepoints = sorted(meta.query(f'Strain=="{select_lineage}"')['Day'].unique())
print(pops)
print(timepoints)

In [ ]:
meta.query(f'population=="{select_lineage}_01"')

In [ ]:
def traceAlleleFreq(pop, min_freq=0.33):
    D = []
    sorted_meta = meta.query(f'population=="{pop}"').sort_values(by='Day')
    for i, row in sorted_meta.iterrows():
        df = pd.read_csv(PROJECT_PATH / f"out/{row['FolderDate']}/{row['source_file']}/output/output.gd.tsv", sep='\t')
        D.append(df)

    # Remove mutations detected in the wild-type background
    # 1. Select background mutations with strong signal
    wt_background=D[0].loc[D[0].frequency>0.01, 'position']
    #print(f"WT background mutations: {wt_background.values}")
    D_=[]

    #print(f"# of mutations\t# of (freq>{min_freq}):")
    for i in range(len(D)):
        # 2. Filter out mutations by position on the chromosome
        df = D[i][~D[i]['position'].isin(wt_background)].copy()
        df['Experiment_Timepoint'] = i
        D_.append(df)
        #print(f"{df.shape[0]}\t{sum(df['frequency']>min_freq)}")
    all_mutations = pd.concat(D_)

    # Filter out mutations with frequency below the threshold
    all_mutations = all_mutations[all_mutations['frequency'] > min_freq]
    all_mutations.sort_values(by=['Experiment_Timepoint'],inplace=True)

    # Fill missing timepoints with 0
    all_mutations.fillna({'aa_ref_seq': '',
            'aa_new_seq': '',
            'aa_pos': '',
            'ref_seq': '',
            'new_seq': ''
            },
            inplace=True)
    
    T = all_mutations.groupby(['position', 'ref_seq', 'new_seq', 'aa_new_seq']).agg(
        freq=('frequency', list),
        timepoints=('Experiment_Timepoint', list),
        gene_name=('gene_name', 'first'),
        gene_product=('gene_product', 'first'),
        aa_ref_seq=('aa_ref_seq', 'first'),
        codon_ref_seq=('codon_ref_seq', 'first'),
        aa_pos=('aa_position', 'first'),
        gene_pos=('gene_position', 'first'),
        mut_cat=('mutation_category', 'first')
    ).reset_index(names=['position', 'ref_seq', 'new_seq', 'aa_new_seq'])

    # Fill the frequency vector with 0 for missing timepoints
    for i, row in T.iterrows():
        for j in range(len(timepoints)):
            if j not in row['timepoints']:
                row['freq'].insert(j, 0)
                row['timepoints'].insert(j, j)
    
    # Drop timepoints
    T.drop(columns=['timepoints'], inplace=True)
    
    # Round all frequency values to 2 decimal places
    T['freq'] = T['freq'].apply(lambda x: [round(i, 2) for i in x])



    # Create labels for each mutation
    nsi = pd.isna(T['aa_pos'])
    T.loc[nsi,'label'] = T.loc[nsi,'gene_name'] + ' ' + T.loc[nsi, 'mut_cat']
    T.loc[~nsi, 'label'] = T.loc[~nsi,'gene_name'] + ' ' + T.loc[~nsi,'aa_ref_seq'] + T.loc[~nsi, 'aa_pos'].astype(str).str.replace(r'\.0','',regex=True) + T.loc[~nsi,'aa_new_seq']
    T.sort_values(by=['mut_cat','gene_name'],ascending=False,inplace=True)
    T.reset_index(inplace=True,drop=True)

    print(f"Population {pop} # of tracked mutations: {T.shape[0]}")

    return T

In [ ]:
# def traceAlleleFreq(pop, min_freq=0):

#     D = []
#     sorted_meta = meta.query(f'population=="{pop}"').sort_values(by='Day')
#     for i, row in sorted_meta.iterrows():
#         df = pd.read_csv(PROJECT_PATH / f"out/{row['FolderDate']}/{row['source_file']}/output/output.gd.tsv", sep='\t')
#         D.append(df)

#     # Remove mutations detected in the wild-type background
#     # 1. Select background mutations with strong signal
#     wt_background=D[0].loc[D[0].frequency>0.01, 'position']
#     #print(f"WT background mutations: {wt_background.values}")
#     D_=[]

#     #print(f"# of mutations\t# of (freq>{min_freq}):")
#     for i in range(len(D)):
#         # 2. Filter out mutations by position on the chromosome
#         df=D[i][~D[i]['position'].isin(wt_background)]
#         D_.append(df)
#         #print(f"{df.shape[0]}\t{sum(df['frequency']>min_freq)}")
    
#     tracked_positions=[]
#     for i in range(len(D_)):
#         d=D_[i]
#         tracked_positions.extend(list(d.loc[ (d['frequency']>min_freq), 'position' ])) #& ~d['mutation_category'].isin(['mobile_element_insertion','large_deletion']), 'position' ] ))
#     tracked_positions=set(tracked_positions)
#     print(f"Population {pop} # of tracked mutations: {len(tracked_positions)}")

#     ## 3. Trace the frequency of those mutations
#     T=[]
#     for pos in tracked_positions:

#         freq=[]
#         for i in range(len(D_)):
#             d = D_[i]
#             ind = d['position']==pos
            
#             if sum(ind)<1:
#                 freq.append(0)
#             else:
#                 freq.append( (d.loc[ind , 'frequency'].values)[0] )
#                 gene_name = d.loc[ind, 'gene_name'].values[0]
#                 gene_product = d.loc[ind, 'gene_product'].values[0]
#                 aa_ref_seq = d.loc[ind, 'aa_ref_seq'].values[0]
#                 aa_new_seq = d.loc[ind, 'aa_new_seq'].values[0]
#                 new_seq = d.loc[ind, 'new_seq'].values[0]
#                 codon_ref_seq = d.loc[ind, 'codon_ref_seq'].values[0]
#                 aa_pos = d.loc[ind, 'aa_position'].values[0]
#                 gene_pos = d.loc[ind, 'gene_position'].values[0]
#                 mut_cat = d.loc[ind, 'mutation_category'].values[0]

#         T.append({'position': pos, 'freq': np.round(freq,2), 
#                 'gene_name':gene_name, 'gene_product':gene_product,
#                 'aa_ref_seq':aa_ref_seq, 'aa_new_seq':aa_new_seq,
#                 'aa_pos': aa_pos, 'gene_pos':gene_pos, 'mut_cat': mut_cat,
#                 'new_seq': new_seq, 'codon_ref_seq': codon_ref_seq})

#     T=pd.DataFrame(T)
#     T.fillna({'aa_ref_seq': '',
#               'aa_new_seq': '',
#               'aa_pos': ''},inplace=True)
    
#     nsi = T['aa_pos']==''
#     T.loc[nsi,'label'] = T.loc[nsi,'gene_name'] + ' ' + T.loc[nsi, 'mut_cat']
#     T.loc[~nsi, 'label'] = T.loc[~nsi,'gene_name'] + ' ' + T.loc[~nsi,'aa_ref_seq'] + T.loc[~nsi, 'aa_pos'].astype(str).str.replace(r'\.0','',regex=True) + T.loc[~nsi,'aa_new_seq']
#         # T.loc[nsi,'gene_pos'] + ' '
#     T.sort_values(by=['mut_cat','gene_name'],ascending=False,inplace=True)
#     T.reset_index(inplace=True,drop=True)
    
#     return T

In [ ]:
# def traceAlleleFreq(pop, min_freq=.33):
#     D = []
#     sorted_meta = meta.query(f'population=="{pop}"').sort_values(by='Day')
#     for i, row in sorted_meta.iterrows():
#         df = pd.read_csv(PROJECT_PATH / f"out/{row['FolderDate']}/{row['source_file']}/output/output.gd.tsv", sep='\t')
        
#         # Generate unique row numbers based on specified columns
#         unique_columns = ['aa_new_seq', 'aa_position', 'gene_name', 'position', 'type']
#         unique_labels, _ = pd.factorize(df[unique_columns].apply(tuple, axis=1))
#         df['unique_row_number'] = unique_labels + 1  # Start numbering from 1
        
#         D.append(df)

#     # Remove mutations detected in the wild-type background
#     wt_background = D[0].loc[D[0].frequency > 0.01, 'unique_row_number']
#     D_ = []

#     for i in range(len(D)):
#         # Filter out mutations based on unique_row_number instead of position
#         df = D[i][~D[i]['unique_row_number'].isin(wt_background)]
#         D_.append(df)

#     tracked_positions = []
#     for i in range(len(D_)):
#         d = D_[i]
#         # Use unique_row_number instead of position for tracking
#         tracked_positions.extend(list(d.loc[d['frequency'] > min_freq, 'unique_row_number']))
#     tracked_positions = set(tracked_positions)
#     print(f"Population {pop} # of tracked mutations: {len(tracked_positions)}")

#     ## 3. Trace the frequency of those mutations
#     T = []
#     for unique_id in tracked_positions:
#         freq = []
#         for i in range(len(D_)):
#             d = D_[i]
#             ind = d['unique_row_number'] == unique_id
            
#             if sum(ind) < 1:
#                 freq.append(0)
#             else:
#                 freq.append(d.loc[ind, 'frequency'].values[0])
#                 gene_name = d.loc[ind, 'gene_name'].values[0]
#                 gene_product = d.loc[ind, 'gene_product'].values[0]
#                 aa_ref_seq = d.loc[ind, 'aa_ref_seq'].values[0]
#                 aa_new_seq = d.loc[ind, 'aa_new_seq'].values[0]
#                 new_seq = d.loc[ind, 'new_seq'].values[0]
#                 codon_ref_seq = d.loc[ind, 'codon_ref_seq'].values[0]
#                 aa_pos = d.loc[ind, 'aa_position'].values[0]
#                 gene_pos = d.loc[ind, 'gene_position'].values[0]
#                 mut_cat = d.loc[ind, 'mutation_category'].values[0]

#         T.append({
#             'unique_row_number': unique_id, 'freq': np.round(freq, 2), 
#             'gene_name': gene_name, 'gene_product': gene_product,
#             'aa_ref_seq': aa_ref_seq, 'aa_new_seq': aa_new_seq,
#             'aa_pos': aa_pos, 'gene_pos': gene_pos, 'mut_cat': mut_cat,
#             'new_seq': new_seq, 'codon_ref_seq': codon_ref_seq
#         })

#     T = pd.DataFrame(T)
#     T.fillna({'aa_ref_seq': '', 'aa_new_seq': '', 'aa_pos': ''}, inplace=True)

#     nsi = T['aa_pos'] == ''
#     T.loc[nsi, 'label'] = T.loc[nsi, 'gene_name'] + ' ' + T.loc[nsi, 'mut_cat']
#     T.loc[~nsi, 'label'] = T.loc[~nsi, 'gene_name'] + ' ' + T.loc[~nsi, 'aa_ref_seq'] + T.loc[~nsi, 'aa_pos'].astype(str).str.replace(r'\.0', '', regex=True) + T.loc[~nsi, 'aa_new_seq']
    
#     T.sort_values(by=['mut_cat', 'gene_name'], ascending=False, inplace=True)
#     T.reset_index(drop=True, inplace=True)
    
#     return T


In [ ]:
af=[]
for pop in pops:
    # print(traceAlleleFreq(pop, min_freq=0.2).shape)
    af.append(traceAlleleFreq(pop, min_freq=0.1))
    af[-1].to_csv(f"data/genomics/data/processed/traced_alleles/{select_lineage}/{pop}.csv", index=False)

In [ ]:
pd.set_option("display.max_rows", 100)
af[5].head(100)

In [ ]:
for index, dataframe in enumerate(af):
    print(f"DataFrame {index} has {len(dataframe)} rows.")

In [ ]:
for index, dataframe in enumerate(af):
    print(f"DataFrame {index} has {len(dataframe)} rows.")

In [ ]:
####### This code fixes the error in output regarding selB. This has been validated by inspecting the evidence files where a TG is inserted, but seperate T and a G insertions with identical frequencies is called.
####### This code removes the duplicate and renames the insertion to the correct value

for n in range(10): # iterate over the dataframes
    df = af[n]
    selb = df.query('gene_name=="selB"')
    if len(selb)<1:
        continue
    vects = np.vstack([freq for freq in selb.freq.values])
    combined_freq = vects.max(axis=0)
    first_row = selb.iloc[0,:].copy()
    first_row['new_seq'] = 'TG'
    first_row['freq'] = combined_freq
    af[n] = af[n].drop(selb.index)
    af[n] = pd.concat([af[n], first_row.to_frame().T], ignore_index=True)
    
    
    # # Update new_seq from 'G' to 'TG' where gene_name is 'selB' and new_seq is 'G'
    # af[n].loc[(af[n]['gene_name'] == 'selB') & (af[n]['new_seq'] == 'G'), 'new_seq'] = 'TG'


In [ ]:
#this code accounts for a label feature that gave an identical label to two different indels of ftsH. the first is a deletion,the second an insertion. so they are labelled accordingly and can be tracked.
# the deletion occured in nearly all cultures, so the label will be left as is. only the label for the insetion in culture 6 with be changed

af[5].loc[
    (af[5]['gene_name'] == 'ftsH') & (af[5]['ref_seq'] == 'G'),
    'label'
] = 'ftsH small_ins'

In [ ]:
clt = 6

selB_rows = af[clt-1].query('gene_name=="ftsH"')

# Display the filtered rows
selB_rows

In [ ]:


# Assuming af is a list of DataFrames
compiled_unique_labels = set()  # Use a set to avoid duplicates

# Loop through the list of DataFrames
for i in range(10):  # Iterate over af[0] to af[9]
    df = af[i]
    
    if isinstance(df, pd.DataFrame):
        # Filter for rows where 'gene_name' is 'fusA'
        filtered_df = df[df['gene_name'] == 'mutL']
        
        # Get the unique labels and update the set
        unique_labels = filtered_df['label'].unique()
        compiled_unique_labels.update(unique_labels)
    else:
        print(f"Error: af[{i}] is not a DataFrame.")

# Convert the set to a sorted list (optional)
compiled_unique_labels_list = sorted(list(compiled_unique_labels))

print(compiled_unique_labels_list)

# 
# ['fusA A678V', 'fusA F593L', 'fusA F605L', 'fusA G117C', 'fusA G46C', 'fusA I61M', 'fusA P610L', 'fusA P610T', 'fusA V116F']
# ['fusA A608V', 'fusA A678V', 'fusA F593L', 'fusA F605L', 'fusA G117C', 'fusA G46C', 'fusA I61M', 'fusA P610L', 'fusA P610T', 'fusA R59H', 'fusA V116F']



In [ ]:
common = np.concatenate([df.label.unique() for df in af])
common, counts = np.unique(common, return_counts=True)
common = pd.DataFrame(dict(zip(common, counts)), index=[0]).melt().sort_values(by='value', ascending=False)
common = common.query('value>2')

f, ax = plt.subplots(1, 1, figsize=(24, 4))
sns.barplot(x=common['variable'], y=common['value'], hue=common['variable'], palette="deep", ax=ax)
# rotate x axis labels 90 degrees
plt.xticks(rotation=90);
plt.xticks(fontsize=20);
#ax3.set_ylabel("Qualitative")
# common
# f.savefig(PROJECT_PATH / "figures/mutation_counts/PLACmutations.png", dpi=300, bbox_inches='tight')

In [ ]:
# import itertools
# # 9 member clade
# select_mutations = np.array(['gyrA S83L','trkH L185Q','glvC small_indel','rpoZ small_indel',
#                     'hipA P86L', 'selB small_indel', 'fimB/fimE mobile_element_insertion',
#                     'ftsH small_indel', 'gatA mobile_element_insertion','mutL small_indel'])

# # What happened to PL
# # select_mutations = np.array(['icd H366H','rpoB D1064G', 'rpoB D1064N', 'rpoB E562D', 'rpoB H1237L',
# #                               'rpoB M129R', 'rpoB small_indel','gyrA A119E', 'gyrA D87Y', 
# #                               'gyrA G81D', 'gyrA S83L', 'gyrA S83W'])
# # # GyrA
# # select_mutations = np.array(['gyrA A119E', 'gyrA D87Y', 
# #                             'gyrA G81D', 'gyrA S83L', 'gyrA S83W'])
# #rpoB
# # select_mutations = np.array(['rpoB D1064G', 'rpoB D1064N', 'rpoB E562D', 'rpoB H1237L',
# #                                 'rpoB M129R', 'rpoB small_indel'])

# # # PA trkH FusA
# # select_mutations = np.array(['trkH G156C', 'trkH L185Q', 'trkH L80Q', 'trkH Q159H', 'trkH S105Y',
# #                        'trkH T20P', 'trkH V155E', 'trkH small_indel', 'fusA P610L'])


# # PLAC
# # trkH
# #select_mutations = np.array(['trkH L185Q', 'trkH L80Q', 'trkH T20I'])


# #fusA
# #select_mutations = np.array(['fusA A608V', 'fusA A678V', 'fusA F593L', 'fusA F605L', 'fusA G117C', 'fusA G46C', 'fusA I61M', 'fusA P610L', 'fusA P610T', 'fusA R59H', 'fusA V116F'])




# palet = sns.color_palette("muted", n_colors=12)
# select_mut_colors = {val:palet[key] for key, val in enumerate(select_mutations)}

# def plotAlleleFreq(T, ax, x_labels_on=True, title=None, legend_loc=None):

#     # palette = itertools.cycle(sns.color_palette("deep"))
#     # palette = itertools.cycle(sns.color_palette("hls", 8))
#     # palette = itertools.cycle(sns.color_palette("muted"))
#     z=20
#     #pl.figure(figsize=(12,2),dpi=150)
    
#     x_offset = {lbl:offs for lbl, offs in zip(select_mutations, np.linspace(-0.33,0.33,len(select_mutations)))}
#     # color_dict = {label:palet[cix] for cix, label in enumerate(T.query(f"gene_name.isin({str(select_mutations)})")['label'].unique())}

#     for i in range(T.shape[0]):
#         row_label = T.loc[i, 'label']
#         if row_label in select_mutations:
#             ax.plot(np.arange(len(timepoints))+x_offset[row_label],
#                     T.loc[i,'freq']+np.random.rand(len(timepoints))*0.05,
#                     '.-', color=select_mut_colors[row_label], #color_dict[T.loc[i,'label']],#=
#                     label=T.loc[i,'label'], alpha=0.9, lw=1.5, zorder=z)
#         else:
#             ax.plot(np.arange(len(timepoints)),T.loc[i,'freq'],'.-',alpha=0.1, lw=1, ms=3, c='black', zorder=10)

#     if title:
#         ax.text(-500,0.85,title,fontweight='bold')
#     ax.set_yticks(np.arange(0,1.1,0.2))
#     ax.set_ylabel('frequency')
    
#     major_ticks = np.arange(len(timepoints))
#     ax.set_xticks(major_ticks)
#     ax.set_xticklabels(timepoints)
#     if x_labels_on:
#         ax.set_xlabel('Day')
#         #ax.legend(loc='lower center', bbox_to_anchor=(0.5, -2),ncol=4)
#     else:
#         ax.set_xticklabels(['']*len(major_ticks))
    
#     if legend_loc:
#         # ax.legend(loc='upper center', bbox_to_anchor=(legend_loc[0], legend_loc[1]),ncol=4, fontsize=7)
#         handles, labels = ax.get_legend_handles_labels()
#         order = [labels.index(gene) for gene in select_mutations if gene in labels]
#         ax.legend([handles[idx] for idx in order], [labels[idx] for idx in order],
#                   loc='upper center', bbox_to_anchor=(legend_loc[0], legend_loc[1]), ncol=4, fontsize=7)
#     #ax.set_title(f'Population {pop}')
    
#     #pl.savefig(f'Rise of hyper-mutators.png', dpi=150, bbox_inches='tight')
#     #pl.show()

In [ ]:
# Assign the group variable
group = 'fusA'  # Options: 'clade', 'PC', 'PL', 'GyrA', 'rpoB', 'PA', 'trkH', 'fusA'

# Select mutations based on group
if group == 'clade':
    select_mutations = [
        'gyrA S83L',
        'trkH L185Q',
        'glvC small_indel',
        'rpoZ small_indel',
        'hipA P86L',
        'selB small_indel',
        'fimB/fimE mobile_element_insertion',
        'ftsH small_indel',
        'gatA mobile_element_insertion'
    ]
    alt_labels = {
    'gyrA S83L': 'GyrA S83L',
    'trkH L185Q': 'TrkH L185Q',
    'glvC small_indel': r'$\it{glvC}$ indel',
    'rpoZ small_indel': r'$\it{rpoZ}$ indel',
    'hipA P86L': 'HipA P86L',
    'selB small_indel': r'$\it{selB}$ indel',
    'fimB/fimE mobile_element_insertion': r'$\it{fimB/E}$ IS5 insertion',
    'ftsH small_indel': r'$\it{ftsH}$ indel',
    'gatA mobile_element_insertion': r'$\it{gatA}$ IS5 insertion'
}
elif group == 'PC':
    select_mutations = [
        'cysS Y298S', 'waaF D13E', 'prs A114V', 'glnH A17V', 'prs A76V',
        'cra S18N', 'corA H75N', 'corA L50Q', 'pheS A14P', 'phoQ G39C',
        'mrcB|mrcB T|T702|657S|S', 'mgtL/mgtA small_indel', 'phoQ small_indel',
        'phoQ mobile_element_insertion', 'prs R79C', 'prs I151T', 'rbsA S157R',
        'relB P45S', 'yicC mobile_element_insertion', 'yjbB L14Q'
    ]
elif group == 'PL':
    select_mutations = [
        'icd H366H', 'rpoB D1064G', 'rpoB D1064N', 'rpoB E562D', 'rpoB H1237L',
        'rpoB M129R', 'rpoB small_indel', 'gyrA A119E', 'gyrA D87Y',
        'gyrA G81D', 'gyrA S83L', 'gyrA S83W'
    ]
elif group == 'gyrA':
    select_mutations = [
        'gyrA A119E', 'gyrA D87Y', 'gyrA G81D', 'gyrA S83L', 'gyrA S83W'
    ]
elif group == 'rpoB':
    select_mutations = [
        'rpoB D1064G', 'rpoB D1064N', 'rpoB E562D', 'rpoB H1237L',
        'rpoB M129R', 'rpoB small_indel']
    # , 'icd H366H']
    alt_labels = {
    'rpoB small_indel': 'rpoB indel',
}
elif group == 'PA':
    select_mutations = [
        'trkH G156C', 'trkH L185Q', 'trkH L80Q', 'trkH Q159H', 'trkH S105Y',
        'trkH T20P', 'trkH V155E', 'trkH small_indel', 'fusA P610L'
    ]
elif group == 'trkH':
    select_mutations = [
        'trkH L185Q', 'trkH L80Q', 'trkH T20I'
    ]
elif group == 'fusA':
    select_mutations = [
        'fusA A608V', 'fusA A678V', 'fusA F593L', 'fusA F605L', 'fusA G117C',
        'fusA G46C', 'fusA I61M', 'fusA P610L', 'fusA P610T', 'fusA R59H', 'fusA V116F'
    ]
else:
    raise ValueError("Invalid group name. Please assign a valid group.")



palet = sns.color_palette("muted", n_colors=len(select_mutations))
select_mut_colors = {val:palet[key] for key, val in enumerate(select_mutations)}



In [ ]:
import itertools
#clade
select_mutations = np.array([
    'gyrA S83L',
    'trkH L185Q',
    'glvC small_indel',
    'rpoZ small_indel',
    'hipA P86L',
    'selB small_indel',
    'fimB/fimE mobile_element_insertion',
    'ftsH small_indel',
    'gatA mobile_element_insertion'
])

# Alternative labels dictionary
# alt_labels = {
#     'gyrA S83L': 'gyrA S83L',
#     'trkH L185Q': 'trkH L185Q',
#     'glvC small_indel': 'glvC indel',
#     'rpoZ small_indel': 'rpoZ indel',
#     'hipA P86L': 'hipA P86L',
#     'selB small_indel': 'selB indel',
#     'fimB/fimE mobile_element_insertion': 'fimB/E IS5 insertion',
#     'ftsH small_indel': 'ftsH indel',
#     'gatA mobile_element_insertion': 'gatA IS5 insertion'
# }

# PC
# select_mutations = np.array(['cysS Y298S', 'waaF D13E', 'prs A114V', 'glnH A17V', 'prs A76V', 
#                                   'cra S18N', 'corA H75N', 'corA L50Q', 'pheS A14P', 'phoQ G39C', 
#                                   'mrcB|mrcB T|T702|657S|S', 'mgtL/mgtA small_indel', 'phoQ small_indel',
#                                     'phoQ mobile_element_insertion', 'prs R79C', 'prs I151T', 'rbsA S157R', 
#                                     'relB P45S', 'yicC mobile_element_insertion', 'yjbB L14Q'])

# PL
# select_mutations = np.array(['icd H366H','rpoB D1064G', 'rpoB D1064N', 'rpoB E562D', 'rpoB H1237L',
#                               'rpoB M129R', 'rpoB small_indel','gyrA A119E', 'gyrA D87Y', 
#                               'gyrA G81D', 'gyrA S83L', 'gyrA S83W'])
# GyrA
# select_mutations = np.array(['gyrA A119E', 'gyrA D87Y', 
#                             'gyrA G81D', 'gyrA S83L', 'gyrA S83W'])
# rpoB
# select_mutations = np.array(['rpoB D1064G', 'rpoB D1064N', 'rpoB E562D', 'rpoB H1237L',
#                                 'rpoB M129R', 'rpoB small_indel'])

# # PA
# select_mutations = np.array(['trkH G156C', 'trkH L185Q', 'trkH L80Q', 'trkH Q159H', 'trkH S105Y',
#                        'trkH T20P', 'trkH V155E', 'trkH small_indel', 'fusA P610L'])

# trkH
# select_mutations = np.array(['trkH L185Q', 'trkH L80Q', 'trkH T20I'])


#fusA
#select_mutations = np.array(['fusA A608V', 'fusA A678V', 'fusA F593L', 'fusA F605L', 'fusA G117C', 'fusA G46C', 'fusA I61M', 'fusA P610L', 'fusA P610T', 'fusA R59H', 'fusA V116F'])




palet = sns.color_palette("muted", n_colors=len(select_mutations))
select_mut_colors = {val:palet[key] for key, val in enumerate(select_mutations)}

In [ ]:

### Adam modified to make 1 legend, 1 x-axis label


def plotAlleleFreq(T, ax, title=None, show_shared_legend=False):
    import matplotlib.pyplot as plt
    import matplotlib.lines as mlines  # Ensure this import is included
    
    z = 20
    x_offset = {lbl: offs for lbl, offs in zip(select_mutations, np.linspace(-0.2, 0.2, len(select_mutations)))}
    all_handles = []
    all_labels = []

    for i in range(T.shape[0]):
        row_label = T.loc[i, 'label']
        if row_label in select_mutations:
            line, = ax.plot(
                np.arange(len(timepoints)) + x_offset[row_label],
                T.loc[i, 'freq'] + np.random.rand(len(timepoints)) * 0.05,
                '.-', color=select_mut_colors[row_label],
                label=row_label, alpha=0.9, lw=4, zorder=z
            )
            # Collect handle and label only if unique
            if row_label not in all_labels:
                all_handles.append(line)
                all_labels.append(row_label)
        else:
            ax.plot(
                np.arange(len(timepoints)), T.loc[i, 'freq'], '.-',
                alpha=0.1, lw=1, ms=3, c='black', zorder=10
            )

    if title:
        ax.text(-500, 0.85, title, fontweight='bold')
    yticks = np.arange(0, 1.1, 0.2)
    ax.set_yticks(yticks)  # Set the tick positions
    ax.set_yticklabels([f"{y:.1f}" for y in yticks], fontsize=16) 
    ax.set_ylabel('Frequency', fontsize= 16)
    ax.set_ylabel('Frequency', color = 'white')
    ax.tick_params(axis='y')  # Set y-tick labels to white

  

    major_ticks = np.arange(len(timepoints))
    ax.set_xticks(major_ticks)
    ax.set_xticklabels(timepoints)


    # Display single legend if show_shared_legend is True
    if show_shared_legend:
        fig = plt.gcf()  # Get the current figure

        # Dynamically decide whether to use alt_labels
        legend_handles = []
        legend_labels = []
        for mutation in select_mutations:
            label = alt_labels.get(mutation, mutation)  # Use alt_labels if available, else default to mutation
            legend_handles.append(
                mlines.Line2D([], [], color=select_mut_colors[mutation], label=label, lw=1.5)
            )
            legend_labels.append(label)

        fig.legend(
            legend_handles,
            legend_labels,
            loc='upper center',
            bbox_to_anchor=(0.55, 1.1),
            ncol=4,
            fontsize=12,
            columnspacing=0.6,  # Reduce space between columns
            handletextpad=0.4,  # Reduce space between marker and label
            handleheight=1.0,  # Adjust row spacing (marker height)
            handlelength=1.0   # Adjust marker length
        )


In [ ]:
plt.rcParams["font.family"] = "Nimbus Roman"      #big panels
fig, axs = plt.subplots(10, 1, figsize=(5, 15), dpi=150,sharex=True)
for i in range(10):
    plotAlleleFreq(af[i], axs[i],  show_shared_legend=False)
    #axs[i].text(-1.5, .5, f"Culture {i+1}", fontweight='bold', ha='right',fontsize=18)  #comment off/on
    axs[i].set_xlim([-.4, len(timepoints)-1+.4])
    axs[i].set_ylim([-0.1, 1.1])
axs[-1].set_xlabel('Day', fontsize=16)
axs[-1].set_xticks(np.arange(len(timepoints)))
axs[-1].set_xticklabels(timepoints, fontsize=16)
fig.tight_layout()
fig.savefig(PROJECT_PATH / f"figures/final/PLAC_panels_{group}.png", dpi=600, bbox_inches='tight')


In [ ]:
# fig, axs = plt.subplots(10, 1, figsize=(6.75, 15), dpi=150)
# for i in range(10):
#     plotAlleleFreq(af[i], axs[i], x_labels_on=True)
#     axs[i].legend(loc='upper center', bbox_to_anchor=(1.33, 1.05), ncol=1, fontsize=6)
#     axs[i].text(-0.33, .9, f"Culture {i+1:02d}", fontweight='bold')
#     axs[i].set_xlim([-.4, len(timepoints)-1+.4])
#     axs[i].set_ylim([-0.1, 1.1])
# fig.tight_layout()
# fig.savefig(PROJECT_PATH / "figures/tracedalleles/PLA_emergence_trkH.png", dpi=300, bbox_inches='tight')

In [ ]:
# fig, axs = plt.subplots(10, 1, figsize=(6.75, 15), dpi=150)
# for i in range(10):
#     plotAlleleFreq(af[i], axs[i], x_labels_on=True)
#     axs[i].legend(loc='upper center', bbox_to_anchor=(1.33, 1.05), ncol=1, fontsize=6)
#     axs[i].text(-0.33, .9, f"Culture {i+1:02d}", fontweight='bold')
#     axs[i].set_xlim([-.4, len(timepoints)-1+.4])
#     axs[i].set_ylim([-0.1, 1.1])
# fig.tight_layout()
# fig.savefig(PROJECT_PATH / "figures/tracedalleles/PLA_emergence_fusA.png", dpi=300, bbox_inches='tight')

In [ ]:
fig, axs = plt.subplots(10, 1, figsize=(6.75, 15), dpi=150)
for i in range(10):
    plotAlleleFreq(af[i], axs[i])
    axs[i].legend(loc='upper center', bbox_to_anchor=(1.33, 1.05), ncol=1, fontsize=6)
    axs[i].text(-0.33, .9, f"Culture {i+1:02d}", fontweight='bold')
    axs[i].set_xlim([-.4, len(timepoints)-1+.4])
    axs[i].set_ylim([-0.1, 1.1])
fig.tight_layout()
#fig.savefig(PROJECT_PATH / "figures/tracedalleles/PLAC_rpozclade_mutL.png", dpi=300, bbox_inches='tight')

In [ ]:

from matplotlib.patches import FancyBboxPatch
plt.rcParams["font.family"] = "Nimbus Roman"

fig, axes = plt.subplots(2,5,figsize=(9.5,3),dpi=150)
# set common font size and type that should include all x and y labels
# plt.rcParams.update({'font.size': 5, 'font.family': 'sans-serif'})
# set colors
palet = sns.color_palette("muted")
select_mut_colors = {val:palet[key] for key, val in enumerate(select_mutations)}

for i in range(10):
    tracked_mut = af[i].query(f'label.isin({list(select_mutations)})')
    ax = axes.flat[i]
    for ix, row in tracked_mut.iterrows():
        mut_order = len(select_mutations)-1 - np.where(row['label']==select_mutations)[0][0]
        binary = np.where(np.array(row['freq'])>0.)[0]
        width = binary[-1] - binary[0] + .66
        left = binary[0] 
        rect = FancyBboxPatch((left, mut_order - 0.4), width, 0.5, boxstyle="round,pad=0.15", 
                              color=select_mut_colors[row['label']], alpha=0.8)
        ax.add_patch(rect)
        #ax.barh(ix, binary[-1]-binary[0], offset=binary[0], color=select_mut_colors[row['label']], alpha=0.7)
    ax.set_xlim(0-.3, 10-.7)
    ax.set_ylim(-.8, len(select_mutations)-0.33)
    if i > 4:
        ax.set_xticks(np.arange(len(timepoints)))
        ax.set_xticklabels(timepoints, fontsize=8)
        ax.set_xlabel('Day', fontsize=10)
    else:
        ax.set_xticks([])
        ax.set_xticklabels([])

    if i % 5 == 0:
        mapped_labels = [alt_labels.get(label, label) for label in reversed(select_mutations)]
        ax.set_yticks(np.arange(len(select_mutations)))
        ax.set_yticklabels(mapped_labels, fontsize=8)
        # Hide y-tick marks but keep the labels
        ax.tick_params(axis='y', length=0)
    else:
        ax.set_yticks([])
        ax.set_yticklabels([])
    #ax.set_title(f"PLAC_{i+1:02d}",)
    ax.text(.05, len(select_mutations)-9.3, f"Culture {i+1}",
        fontweight='normal', fontsize=10, fontfamily='Nimbus Roman')
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0, hspace=0)

fig.tight_layout()
fig.savefig(PROJECT_PATH / "figures/final/PLAC_emergence_cadence.png", dpi=600, bbox_inches='tight')

In [ ]:
all = []
for ix, df in enumerate(af):
    dff = df.copy()
    dff['Pop'] = ix+1 
    all.append(dff)
combined_df=pd.concat(all)


combined_df['last_freq'] = combined_df['freq'].apply(lambda x: x[-1])
combined_df


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
plt.rcParams["font.family"] = "Nimbus Roman"

# Define the custom order for mutations
custom_order = ['gyrA S83L', 'trkH L185Q', 'glvC small_indel', 'rpoZ small_indel',
                'hipA P86L', 'selB small_indel', 'fimB/fimE mobile_element_insertion',
                'ftsH small_indel', 'gatA mobile_element_insertion']

# Define alternate labels for specific mutations
alternate_labels = {
    'glvC small_indel': 'glvC indel',
    'rpoZ small_indel': 'rpoZ indel',
    'selB small_indel': 'selB indel',
    'fimB/fimE mobile_element_insertion': 'fimB/E IS5 insertion',
    'ftsH small_indel': 'ftsH indel',
    'gatA mobile_element_insertion': 'gatA IS5 insertion'
}

# Filter the data for select_mutations and restructure it into a pivot table
heatmap_data = combined_df[combined_df['label'].isin(custom_order)]
pivot_table = heatmap_data.pivot_table(
    index='Pop', columns='label', values='last_freq'
)

# Reindex the pivot table to ensure all mutations are included
pivot_table = pivot_table.reindex(columns=custom_order)

# Replace NaN values with 0
pivot_table = pivot_table.fillna(0)

# Prepare annotations: Replace 0s with empty strings
annotations = pivot_table.applymap(lambda x: f"{x:.2f}" if x != 0 else "")

# Plot the heatmap
plt.figure(figsize=(8, 10))  # Adjust the figure size
sns.set(font_scale=1.2)      # Adjust font scaling

# Create the heatmap
ax = sns.heatmap(
    pivot_table,
    annot=annotations,                          # Custom annotations
    fmt='',                                     # Prevent double formatting
    cmap="Purples",                             # Use the Purples colormap
    annot_kws={'size': 20, 'ha': 'center', 'va': 'center'},  # Customize annotation style
    cbar=False,                                 # Disable the color bar
    linewidths=0.5,                             # Add grid lines
    linecolor='lightgrey'                       # Use light grey for grid lines
)

# Customize axis labels and title
ax.set_ylabel('Culture Number', fontsize=25)  # Set y-axis label with larger font
ax.set_xlabel('Mutation', fontsize=25)        # Set x-axis label with larger font
ax.set_title('Final Clade Mutation Frequencies', fontsize=25)  # Set title with LaTeX

# Replace x-axis tick labels with alternate labels where applicable
xtick_labels = [alternate_labels.get(label, label) for label in custom_order]
ax.set_xticklabels(xtick_labels, rotation=45, ha='right', fontsize=20)

# Customize spines
for spine in ax.spines.values():
    spine.set_visible(True)  # Make spines visible
    spine.set_linewidth(2)   # Set the border width
    spine.set_color("black") # Set the border color

# Adjust layout and y-axis tick font size
plt.yticks(fontsize=20)
plt.tight_layout()  # Adjust layout to prevent overlap
plt.savefig(PROJECT_PATH / "figures/mutation_heatmaps/1205_PLAC_sorted.png", dpi=300, bbox_inches='tight')

# Display the heatmap
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
plt.rcParams["font.family"] = "Nimbus Roman"

# Define the custom order for mutations
custom_order = ['gyrA S83L', 'trkH L185Q', 'glvC small_indel', 'rpoZ small_indel',
                'hipA P86L', 'selB small_indel', 'fimB/fimE mobile_element_insertion',
                'ftsH small_indel', 'gatA mobile_element_insertion']

# Define alternate labels for specific mutations
alternate_labels = {
    'gyrA S83L': 'GyrA S83L',
    'trkH L185Q': 'TrkH L185Q',
    'glvC small_indel': r'$\it{glvC}$ indel',
    'rpoZ small_indel': r'$\it{rpoZ}$ indel',
    'hipA P86L': 'HipA P86L',
    'selB small_indel': r'$\it{selB}$ indel',
    'fimB/fimE mobile_element_insertion': r'$\it{fimB/E}$ IS5 insertion',
    'ftsH small_indel': r'$\it{ftsH}$ indel',
    'gatA mobile_element_insertion': r'$\it{gatA}$ IS5 insertion'
}

# Filter the data for select_mutations and restructure it into a pivot table
heatmap_data = combined_df[combined_df['label'].isin(custom_order)]
pivot_table = heatmap_data.pivot_table(
    index='Pop', columns='label', values='last_freq'
)

# Reindex the pivot table to ensure all mutations are included
pivot_table = pivot_table.reindex(columns=custom_order)

# Replace NaN values with 0
pivot_table = pivot_table.fillna(0)

# Transpose the pivot table to swap axes (Pops on x-axis, labels on y-axis)
pivot_table = pivot_table.T

# Prepare annotations: Replace 0s with empty strings
annotations = pivot_table.applymap(lambda x: f"{x:.2f}" if x != 0 else "")

# Plot the heatmap
plt.figure(figsize=(12, 8))  # Adjust the figure size
sns.set(font='Nimbus Roman', font_scale=1.2)

# Create the heatmap
ax = sns.heatmap(
    pivot_table,
    annot=annotations,                          # Custom annotations
    fmt='',                                     # Prevent double formatting
    cmap="Greens",                             # Use the Purples colormap
    annot_kws={'size': 20, 'ha': 'center', 'va': 'center'},  # Customize annotation style
    cbar=False,                                 # Disable the color bar
    linewidths=0.5,                             # Add grid lines
    linecolor='lightgrey'                       # Use light grey for grid lines
)

ax.tick_params(axis='both', which='major', labelsize=15)

# Replace y-axis tick labels with alternate labels where applicable
ytick_labels = [alternate_labels.get(label, label) for label in custom_order]
ax.set_yticklabels(ytick_labels, fontsize=20)  # Keep y-axis labels horizontal

# Customize axis labels and title
ax.set_xticklabels(ax.get_xticklabels(), fontsize=20)

ax.set_ylabel('', fontsize=25)         # Set y-axis label with larger font
ax.set_xlabel('Culture Number', fontsize=25)  # Set x-axis label with larger font
#ax.set_title('Common Clade Mutations of MG$^{{\\mathrm{{LEV,AMI,CEF}}}}$', fontsize=25)  # Set title with LaTeX

for spine in ax.spines.values():
    spine.set_visible(True)  # Make spines visible
    spine.set_linewidth(2)   # Set the border width
    spine.set_color("black") # Set the border color



plt.tight_layout()  # Adjust layout to prevent overlap
plt.savefig(PROJECT_PATH / "figures/final/PLAC_final_heatmap.png", dpi=600, bbox_inches='tight')

# Display the heatmap
plt.show()



In [ ]:
import pandas as pd

# Sample structure of combined_df
# combined_df = pd.DataFrame({
#     'Pop': [1, 1, 2, 2, 3, 3, ...],
#     'freq': [[0,1,0,2,0,3,0,4,0,5], [1,1,0,0,0,0,2,0,0,0], ...]
# })

# Initialize a list to store rows for the final DataFrame
rows = []

# Iterate over each unique Pop value (1 through 10 in this case)
for pop in range(1, 11):
    # Filter combined_df by the current Pop value
    pop_df = combined_df[combined_df['Pop'] == pop]

    # Initialize a dictionary to store timepoint counts for the current Pop
    timepoint_counts = {f"Timepoint_{i+1}": 0 for i in range(10)}

    # Iterate over each row in the filtered DataFrame
    for _, row in pop_df.iterrows():
        freq_values = row['freq']  # Get the list of frequency values

        # Increment the count for each timepoint if the frequency is greater than 0
        for i, value in enumerate(freq_values):
            if value > 0:
                timepoint_counts[f"Timepoint_{i+1}"] += 1

    # Add rows to the list for each timepoint for the current Pop
    for timepoint, count in timepoint_counts.items():
        rows.append({'Pop': pop, 'timepoint': timepoint, 'count': count})

# Create the final DataFrame from the rows list
timepoint_counts_df = pd.DataFrame(rows)


# Define the mapping for each timepoint to its corresponding day
timepoint_to_day = {
    1: 0,
    2: 3,
    3: 16,
    4: 19,
    5: 31,
    6: 34,
    7: 47,
    8: 55,
    9: 63,
    10: 81
}

# Extract the timepoint number and map it to the Day values
timepoint_counts_df['Day'] = timepoint_counts_df['timepoint'].apply(lambda x: timepoint_to_day[int(x.split('_')[1])])

# Display the resulting DataFrame
print(timepoint_counts_df)


In [ ]:
import matplotlib.pyplot as plt

# Set general font size for all plot elements
font_size = 14  # Adjust as desired

# Create the plot
plt.figure(figsize=(10, 6))

# Iterate over each unique Culture and plot the line for each one
for culture in timepoint_counts_df['Pop'].unique():
    # Filter the DataFrame for the current Culture
    culture_df = timepoint_counts_df[timepoint_counts_df['Pop'] == culture]
    
    # Sort by Day to ensure lines are drawn in the correct order
    culture_df = culture_df.sort_values(by='Day')
    
    # Plot the line for the current Culture
    plt.plot(culture_df['Day'], culture_df['count'], label=f'Culture {culture}')

# Customize font sizes
plt.xlabel('Day', fontsize=font_size)
plt.ylabel('Count', fontsize=font_size)
plt.title('Counts by Day for Each Culture', fontsize=font_size + 2)
plt.legend(title=None, fontsize=font_size - 2, title_fontsize=font_size)
plt.xticks(fontsize=font_size - 2)
plt.yticks(fontsize=font_size - 2)
plt.grid(True)

# Show the plot
plt.show()


In [ ]:
############# Graph example cultures 
#### must be run before combined_df is made
import itertools
import numpy as np
########
########
########
# Define the mutation_set you want to analyze
mutation_set = 'clade'  # Change to 'fusA', 'trkH', or 'clade' as needed
Population = 4
########
########
########
########
########
# Use if-elif-else to select the corresponding mutations
labelcolor = 'None'
if mutation_set == 'gyrA':
    select_mutations = np.array(['gyrA A119E', 'gyrA D87Y', 
                                 'gyrA G81D', 'gyrA S83L', 'gyrA S83W'])
    alt_labels = {
        '': ''}
elif mutation_set == 'trkH':
    select_mutations = np.array(['trkH L185Q', 'trkH L80Q', 'trkH T20I'])

elif mutation_set == 'fusA':
    select_mutations  = np.array(['fusA A608V', 'fusA A678V',  'fusA G117C', 'fusA G46C', 'fusA I61M', 'fusA P610L','fusA F593L',  'fusA P610T', 'fusA F605L','fusA R59H', 'fusA V116F'])

elif mutation_set == 'clade':
    # Original mutations as they appear in the dataset
    select_mutations = np.array(['gyrA S83L', 'trkH L185Q', 'glvC small_indel', 'rpoZ small_indel',
                                  'hipA P86L', 'selB small_indel', 'fimB/fimE mobile_element_insertion',
                                  'ftsH small_indel', 'gatA mobile_element_insertion'])

    # Alternative labels for better readability in plots
    alt_labels = {
        'gyrA S83L': 'gyrA S83L',
        'trkH L185Q': 'trkH L185Q',
        'glvC small_indel': 'glvC indel',
        'rpoZ small_indel': 'rpoZ indel',
        'hipA P86L': 'hipA P86L',
        'selB small_indel': 'selB indel',
        'fimB/fimE mobile_element_insertion': 'fimB/E IS5 insertion',
        'ftsH small_indel': 'ftsH indel',
        'gatA mobile_element_insertion': 'gatA IS5 insertion'
    }
    use_alt_labels = True
    labelcolor = "#000000"
    
elif mutation_set == 'rpoB' :
    select_mutations = np.array(['rpoB D1064G', 'rpoB D1064N', 'rpoB E562D', 'rpoB H1237L',
                                'rpoB M129R', 'rpoB small_indel'])

elif mutation_set == 'PL_fate' :
    select_mutations = np.array(['icd H366H','rpoB D1064G', 'rpoB D1064N', 'rpoB E562D', 'rpoB H1237L',
                              'rpoB M129R', 'rpoB small_indel','gyrA A119E', 'gyrA D87Y', 
                              'gyrA G81D', 'gyrA S83L', 'gyrA S83W'])
    
elif mutation_set == 'PC_mutations':
    select_mutations = np.array(['cysS Y298S', 'waaF D13E', 'prs A114V', 'glnH A17V', 'prs A76V', 
                                  'cra S18N', 'corA H75N', 'corA L50Q', 'pheS A14P', 'phoQ G39C', 
                                  'mrcB|mrcB T|T702|657S|S', 'mgtL/mgtA small_indel', 'phoQ small_indel',
                                    'phoQ mobile_element_insertion', 'prs R79C', 'prs I151T', 'rbsA S157R', 
                                    'relB P45S', 'yicC mobile_element_insertion', 'yjbB L14Q'])

else:
    raise ValueError(f"Unknown mutation_set: {mutation_set}")

# Dynamically set the number of colors
mutations_need_colors = len(select_mutations)
palette = sns.color_palette("muted", n_colors=mutations_need_colors)

# Display the palette for verification
sns.palplot(palette)
plt.show()

palet = sns.color_palette("muted", n_colors=12)
select_mut_colors = {val:palet[key] for key, val in enumerate(select_mutations)}
select_mutations


In [ ]:

from matplotlib.gridspec import GridSpec
plt.rcParams["font.family"] = "Nimbus Roman"

def plotAlleleFreq(T, ax, x_labels_on=True, title=None, legend_loc=False, use_alt_labels=False, alt_labels=None):

 
    z=20 # stacks on top of the black lines

    x_offset = {lbl:offs for lbl, offs in zip(select_mutations, np.linspace(-0.2,0.2,len(select_mutations)))} # gives a horizantal jitter

    for i in range(T.shape[0]):   # iterates through the number of rows within T, later T will be linked to af - each population will have a different number of rows
        row_label = T.loc[i, 'label'] # retrieves the label from row i
        display_label = alt_labels.get(row_label, row_label) if use_alt_labels else row_label

        if row_label in select_mutations: # works with each row that matters
            ax.plot(np.arange(len(timepoints))+x_offset[row_label],
                    T.loc[i,'freq']+np.random.rand(len(timepoints))*0.0, ## *0 is a vertical jitter
                    '.-', color=select_mut_colors[row_label],
                    label=display_label, alpha=0.9, lw=4,ms=10,zorder=z)
        else:
            ax.plot(np.arange(len(timepoints)),T.loc[i,'freq'],'.-',alpha=0.1, lw=2, ms=4, c='black', zorder=10)

    if title:
        ax.text(-500,0.85,title,fontweight='bold')
    ax.set_yticks(np.arange(0,1.1,0.2))
    ax.set_yticklabels([f"{x:.1f}" for x in np.arange(0, 1.1, 0.2)], fontsize=15)  # Set labels and font size

    ax.set_ylabel('Frequency', fontsize =20)
    
    major_ticks = np.arange(len(timepoints))
    ax.set_xticks(major_ticks)
    ax.set_xticklabels(timepoints, fontsize = 15, color = labelcolor)
    if x_labels_on:
        ax.set_xlabel('Day', fontsize = 20, color = labelcolor)
        #ax.legend(loc='lower center', bbox_to_anchor=(0.5, -2),ncol=4)
    else:
        ax.set_xticklabels(['']*len(major_ticks))
    
    if legend_loc:
        handles, labels = ax.get_legend_handles_labels()
        
        # Ensure order matches select_mutations and use alternative labels if needed
        order = [labels.index(gene) for gene in select_mutations if gene in labels]
        ordered_labels = [alt_labels.get(labels[idx], labels[idx]) if use_alt_labels else labels[idx] for idx in order]

        ax.legend(
            [handles[idx] for idx in order],
            ordered_labels,
            loc='upper center',
            bbox_to_anchor=(legend_loc[0], legend_loc[1]),
            ncol=4,
            fontsize=7
        )
########
## Plotting  
########
## Plotting  
fig = plt.figure(figsize=(10, 3), dpi=150)  # Maintain figure size
gs = GridSpec(1, 2, width_ratios=[3, 1], figure=fig)  # Allocate more space to plot than legend

# Define main plot area and legend area
ax = fig.add_subplot(gs[0])  # Main plot
# legend_ax = fig.add_subplot(gs[1])  # Legend area

# Plot the allele frequencies
i = Population - 1  # Index for the dataset to plot
plotAlleleFreq(af[i], ax, x_labels_on=True, use_alt_labels=True, alt_labels=alt_labels)

# Add culture text and adjust plot limits
ax.text(-0.15, .95, f"Culture {i+1}", fontweight='normal', fontsize=18)
ax.set_xlim([-.2, len(timepoints) - 1 + .2])  # Add room to the left and right
ax.set_ylim([-0.1, 1.1])
# # Add the legend to the dedicated area
# handles, labels = ax.get_legend_handles_labels()
# legend_ax.axis('off')  # Hide legend subplot axes
# legend_ax.legend(handles, labels, loc='center', bbox_to_anchor = (0, .5), fontsize=14)  # Centralize the legend

# Save the figure
fig.tight_layout(w_pad=1)  # Add padding between the plot and legend
fig.savefig(PROJECT_PATH / f"figures/final/traced_allele_Culture_{Population}_{mutation_set}.png", dpi=600, bbox_inches='tight')
plt.show()